# Demonstration of the `gsf` package


This notebook runs **GalacticStructureFinder** on the bundled example galaxy (`tests/sim1`) for the simplest case of **two kinematic components**, and shows the result: the face-on/edge-on moment maps of the named components, and a table of their physical properties.

The first step computes the gravitational potential at every stellar particle by direct two-body summation, which is the slowest part of the run; exporting `OMP_NUM_THREADS` (up to the number of cores on your machine) before starting Jupyter speeds it up considerably.


In [ ]:
import os, glob, pickle
import numpy as np
from IPython.display import Image, HTML, display
import gsf

# Locate the bundled example galaxy (works whether this notebook runs from the
# repository root or from the notebooks/ directory).
sim_dir = next(p for p in ('tests/sim1', '../tests/sim1', '../../tests/sim1')
               if os.path.exists(os.path.join(p, 'sim1.halo_1.star.dat')))
star = os.path.join(sim_dir, 'sim1.halo_1.star.dat')
gas  = os.path.join(sim_dir, 'sim1.halo_1.gas.dat')
dark = os.path.join(sim_dir, 'sim1.halo_1.dark.dat')

out_dir = 'demo_output/'
os.makedirs(out_dir, exist_ok=True)

# Decompose the galaxy into two components. This runs the whole pipeline:
# potential -> derived quantities -> GMM clustering -> moment maps -> naming.
gsf.gsf(star, gas, dark, number_of_clusters=2, out_dir=out_dir, plot=False, verbose=False)


## Component maps

For every component `gsf` produces, from left to right, the face-on surface mass density, the edge-on surface mass density, and the edge-on line-of-sight velocity. The rows are stacked into a single mosaic (the whole galaxy on top, then the components ordered by mass fraction), with each component labelled by the physical name assigned by `tag_components`.


In [ ]:
# The two-inclination mosaic written by tag_components -> plot_2inclinations_moment_maps.
mosaic = sorted(glob.glob(out_dir + '*_2inc_mosaic.png'))[0]
display(Image(filename=mosaic))


## Component properties

`gsf` also writes a per-model summary (`*summary.dat`) with the mass-weighted properties of each component. Below we load it and the component names and lay out the main quantities: mass fraction, the equatorial-plane asymmetry `b/a`, the flatness `xi = 2(1 - c/a) - 1`, and the mass-weighted median (with 16th/84th percentiles) of the 3D radius, rotational velocity, vertical-velocity dispersion, and normalized binding energy.


In [ ]:
summ = pickle.load(open(glob.glob(out_dir + '*summary.dat')[0], 'rb'))
res, gl = summ['result'], summ['result_global']
names = pickle.load(open(glob.glob(out_dir + '*_tags.dat')[0], 'rb'))['tags']

def _s(x):
    return float(np.ravel(np.asarray(x))[0])   # robust scalar extraction

def pm(med, lo, hi, fmt='%.2f'):
    return '%s (+%s / -%s)' % (fmt % med, fmt % (hi - med), fmt % (med - lo))

def quantities(kind, j=None):
    if kind == 'all':
        a, b, c = _s(gl['a']), _s(gl['b']), _s(gl['c'])
        massfrac = 1.0
        get = lambda feat, pp: _s(gl[feat][pp])
    else:
        a, b, c = _s(res['a'][j]), _s(res['b'][j]), _s(res['c'][j])
        massfrac = _s(res['total_mass'][j]) / _s(gl['total_mass'])
        get = lambda feat, pp: _s(res[feat][pp][j])
    return {
        'f = M/M*':      '%.2f' % massfrac,
        'b/a':           '%.2f' % (b / a),
        'flatness xi':   '%.2f' % (2 * (1 - c / a) - 1),
        'r [kpc]':       pm(get('r3', 50), get('r3', 16), get('r3', 84)),
        'v_phi [km/s]':  pm(get('vphi', 50), get('vphi', 16), get('vphi', 84), '%.0f'),
        'sigma(v_z) [km/s]': '%.0f' % ((get('vz', 84) - get('vz', 16)) / 2),
        'e/max|e|':      pm(get('e', 50), get('e', 16), get('e', 84)),
    }

columns = [('All', quantities('all'))] + [(names[j], quantities('comp', j)) for j in range(len(names))]
rows = list(columns[0][1].keys())

th = "style='padding:4px 12px;border-bottom:2px solid #444'"
td = "style='padding:4px 12px;text-align:center'"
tl = "style='padding:4px 12px;text-align:left'"
html = "<table style='border-collapse:collapse;font-family:sans-serif'>"
html += '<tr><th %s>property</th>' % tl + ''.join('<th %s>%s</th>' % (th, n) for n, _ in columns) + '</tr>'
for r in rows:
    html += '<tr><td %s><b>%s</b></td>' % (tl, r)
    html += ''.join('<td %s>%s</td>' % (td, d[r]) for _, d in columns)
    html += '</tr>'
html += '</table>'
HTML(html)


## Full LaTeX table (for publication)

For direct inclusion in a paper, `gsf` also writes the complete, richer summary table as a ready-to-`\input` LaTeX file (`*table.tex`). Its contents:


In [ ]:
tex = glob.glob(out_dir + '*table.tex')[0]
print('Written to:', tex, '\n')
print(open(tex).read())
